In [0]:
dbutils.widgets.text("sas_token", "", "Azure Storage SAS Token")

In [0]:
import requests
import json
import xml.etree.ElementTree as ET
from pyspark.sql.functions import col, current_timestamp
import os

def get_sas_token() -> str:
    token = os.getenv("AZURE_SAS_TOKEN", "").strip()
    
    if not token:
        token = dbutils.widgets.get("sas_token").strip()
        
    if not token:
        raise ValueError(
            "SAS Token not provided. Please fill in the 'Azure Storage SAS Token' widget "
            "at the top of the notebook or set the AZURE_SAS_TOKEN environment variable."
        )
        
    return token.lstrip("?")

storage_account = "stdataengprojectrafael"
container = "bronze"
prefix = "world-bank/countries/2026-11-08/"
sas_token = get_sas_token()

list_url = f"https://{storage_account}.blob.core.windows.net/{container}?restype=container&comp=list&prefix={prefix}&{sas_token}"
resp_list = requests.get(list_url)
resp_list.raise_for_status()

root = ET.fromstring(resp_list.content)
blob_names = [blob.find("Name").text for blob in root.findall(".//Blob")]
print("Files found:", blob_names)

all_records = []
for blob_name in blob_names:
    blob_url = f"https://{storage_account}.blob.core.windows.net/{container}/{blob_name}?{sas_token}"
    resp_data = requests.get(blob_url)
    resp_data.raise_for_status()
    
    lines = [json.loads(line) for line in resp_data.text.strip().splitlines() if line.strip()]
    if len(lines) == 1 and isinstance(lines[0], list):
        records = lines[0][1]
    elif len(lines) >= 2 and isinstance(lines[1], list):
        records = lines[1]
    else:
        records = lines
        
    all_records.extend(records)

print(f"Total of unified records: {len(all_records)}")

df_raw_countries = spark.createDataFrame(all_records)
df_raw_countries.printSchema()

In [0]:
df_silver_countries = (
    df_raw_countries
    .select(
        col("id").alias("country_code"),
        col("iso2Code").alias("iso2_code"),
        col("name").alias("country_name"),
        col("capitalCity").alias("capital_city"),
        col("region.id").alias("region_id"),
        col("region.value").alias("region_name"),
        col("incomeLevel.id").alias("income_level_id"),
        col("incomeLevel.value").alias("income_level"),
        col("lendingType.value").alias("lending_type"),
        col("longitude").try_cast("double").alias("longitude"),
        col("latitude").try_cast("double").alias("latitude"),
        current_timestamp().alias("silver_processed_at")
    )
    .filter(col("region_name") != "Aggregates")
)

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

(
    df_silver_countries.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.worldbank_countries")
)

display(spark.sql("SELECT * FROM silver.worldbank_countries LIMIT 10"))